In [22]:
# # Notebook 03 — RAG + Qdrant Retrieval Tests
# 
# This notebook tests:
# - Embedding quality
# - Chunking strategies
# - ATS rule retrieval
# - Similarity thresholds
# - Query reformulation

# %%
# %% 
import sys
import os

# Add project root (WHATSAPP_AGENT) to Python path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("PYTHONPATH updated:", PROJECT_ROOT)

from qdrant_client import QdrantClient
from qdrant_client.http import models as qmodels

from app.config import settings
from app.rag.retriever import embed_text, get_qdrant_client, retrieve_ats_constraints
from typing import Any, Dict, List, Tuple

COLLECTION_NAME = settings.QDRANT_COLLECTION_NAME

qdrant: QdrantClient = get_qdrant_client()



PYTHONPATH updated: c:\Users\gyash\Desktop\virasat\Whatsapp_agent


In [24]:
# %% [markdown]
# # Ingestion Section — Populate Qdrant Before Running Retrieval Tests
# This ensures Notebook 3 has data to test against.

# %%
from app.rag.indexer import create_collection_if_missing, index_ats_rules

# 1) Ensure collection exists with correct vector size for MiniLM (384)
create_collection_if_missing(vector_size=384)

# 2) Define a small set of ATS rules for testing retrieval
test_rules = [
    {
        "rule_text": "Avoid keyword stuffing in your resume. Overusing the same keyword can be penalized by ATS.",
        "embedding": embed_text("Avoid keyword stuffing in your resume. Overusing the same keyword can be penalized by ATS."),
        "category": "keywords",
    },
    {
        "rule_text": "Job titles should be written in a standard format so ATS can correctly parse your experience.",
        "embedding": embed_text("Job titles should be written in a standard format so ATS can correctly parse your experience."),
        "category": "job_titles",
    },
    {
        "rule_text": "ATS systems may not parse images or graphics. Avoid using icons, logos, or decorative elements.",
        "embedding": embed_text("ATS systems may not parse images or graphics. Avoid using icons, logos, or decorative elements."),
        "category": "formatting",
    },
    {
    "rule_text": "Use action verbs at the start of each bullet to improve ATS parsing.",
    "embedding": embed_text("Use action verbs at the start of each bullet to improve ATS parsing."),
    "category": "writing_style",
},
{
    "rule_text": "Keep resume length to one or two pages for optimal ATS scanning.",
    "embedding": embed_text("Keep resume length to one or two pages for optimal ATS scanning."),
    "category": "length",
},
{
    "rule_text": "Save your resume as a PDF unless the job posting specifies otherwise.",
    "embedding": embed_text("Save your resume as a PDF unless the job posting specifies otherwise."),
    "category": "file_format",
},

]

# 3) Index the rules
index_ats_rules(test_rules)

# 4) Verify ingestion
collection_info = qdrant.get_collection(COLLECTION_NAME)
print(f"Collection '{COLLECTION_NAME}' now has {collection_info.points_count} points")


Collection 'ats_parser_rules' now has 6 points


In [25]:
import sys
from qdrant_client import QdrantClient

# 1. Force Python to flush the old config cache out of memory
if 'app.config' in sys.modules:
    del sys.modules['app.config']

# 2. Re-import the fresh settings pointing to 127.0.0.1
from app.config import settings
from app.rag.retriever import get_qdrant_client

# 3. Re-build the client using your helper function
qdrant = get_qdrant_client()

print(f"Client reinitialized. Target URL: {settings.QDRANT_URL}")

# %%
# Qdrant health + collection existence check

try:
    collections = qdrant.get_collections().collections
    names = [c.name for c in collections]
    print("Qdrant is running.")
    print("Collections:", names)
    if COLLECTION_NAME not in names:
        print(f"⚠ Collection '{COLLECTION_NAME}' does NOT exist yet.")
    else:
        print(f"✅ Collection '{COLLECTION_NAME}' is present.")
except Exception as e:
    print("❌ Qdrant connection failed:", e)


Client reinitialized. Target URL: http://127.0.0.1:6333
Qdrant is running.
Collections: ['ats_parser_rules']
✅ Collection 'ats_parser_rules' is present.


In [26]:
# %%
def search_qdrant_vector(
    query_text: str,
    top_k: int = 5,
    score_threshold: float | None = None,
) -> List[Tuple[float, Dict[str, Any]]]:
    """Use the same embedding as retriever, but expose raw Qdrant hits using modern Query API."""
    vec = embed_text(query_text)
    
    # Using the updated query_points API
    response = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=vec,                     # In query_points, query accepts the raw vector array
        limit=top_k,
        with_payload=True,
        score_threshold=score_threshold,
    )
    
    # response.points contains the list of matched hits
    return [(p.score, p.payload or {}) for p in response.points]


def pretty_print_results(results: List[Tuple[float, Dict[str, Any]]], max_chars: int = 200):
    for i, (score, payload) in enumerate(results, start=1):
        text = (payload.get("rule_text") or payload.get("text") or "")[:max_chars]
        text = text.replace("\n", " ")
        print(f"{i}. score={score:.4f} | {text!r}")

In [27]:
# %%
# First, check if collection exists and has data
try:
    collection_info = qdrant.get_collection(COLLECTION_NAME)
    print(f"✅ Collection '{COLLECTION_NAME}' exists with {collection_info.points_count} points")
except Exception as e:
    print(f"⚠ Collection does not exist: {e}")
    print("Please ensure the collection is created and populated first.")
    embedding_failures = []
else:
    embedding_test_queries = [
        # (query, must_contain_any_of)
        (
            "What are the ATS rules for keyword stuffing?",
            ["keyword", "stuffing", "overuse"],
        ),
        (
            "How should resumes handle job titles for ATS?",
            ["job title", "title normalization", "titles"],
        ),
    ]

    def test_embedding_quality():
        failures = []
        for query, expected_terms in embedding_test_queries:
            results = search_qdrant_vector(query, top_k=5)
            joined = " ".join(
                (p.get("rule_text") or p.get("text") or "")
                for _, p in results
            )
            if not any(term.lower() in joined.lower() for term in expected_terms):
                failures.append(
                    {
                        "query": query,
                        "expected_terms": expected_terms,
                        "snippet": joined[:300],
                    }
                )
        return failures

    embedding_failures = test_embedding_quality()
    print("Embedding quality failures:", len(embedding_failures))
    embedding_failures


✅ Collection 'ats_parser_rules' exists with 6 points
Embedding quality failures: 0


In [28]:
# %%
# Inspect a few random chunks / rules
points, _ = qdrant.scroll(
    collection_name=COLLECTION_NAME,
    limit=5,
    with_payload=True,
)

for p in points:
    payload = p.payload or {}
    text = payload.get("rule_text") or payload.get("text") or ""
    print(
        f"id={p.id} | len(text)={len(text)} | keys={list(payload.keys())}"
    )


id=0 | len(text)=90 | keys=['rule_text', 'embedding', 'category']
id=1 | len(text)=93 | keys=['rule_text', 'embedding', 'category']
id=2 | len(text)=95 | keys=['rule_text', 'embedding', 'category']
id=3 | len(text)=68 | keys=['rule_text', 'embedding', 'category']
id=4 | len(text)=64 | keys=['rule_text', 'embedding', 'category']


In [29]:
# %%
def get_chunk_length_stats(sample_size: int = 200):
    points, _ = qdrant.scroll(
        collection_name=COLLECTION_NAME,
        limit=sample_size,
        with_payload=True,
    )
    lengths = [
        len((p.payload or {}).get("rule_text") or (p.payload or {}).get("text") or "")
        for p in points
    ]
    if not lengths:
        return {}
    return {
        "count": len(lengths),
        "min": min(lengths),
        "max": max(lengths),
        "avg": sum(lengths) / len(lengths),
    }

chunk_stats = get_chunk_length_stats()
chunk_stats


{'count': 6, 'min': 64, 'max': 95, 'avg': 79.83333333333333}

In [30]:
# %%
ats_rule_queries = [
    # (user_query, expected_hint_substring)
    (
        "Can I use images or graphics in my resume for ATS?",
        "image",  # expect rule mentioning images/graphics
    ),
    (
        "What file formats are safe for ATS parsing?",
        "format",  # expect rule mentioning file formats
    ),
    (
        "How should I handle bullet points for ATS?",
        "bullet",  # expect rule mentioning bullet points
    ),
]

def test_ats_rule_retrieval():
    failures = []
    for query, expected_substring in ats_rule_queries:
        result = retrieve_ats_constraints(query, top_k=5)
        joined = " ".join(result.rules)
        if expected_substring.lower() not in joined.lower():
            failures.append(
                {
                    "query": query,
                    "expected_substring": expected_substring,
                    "snippet": joined[:300],
                }
            )
    return failures

ats_failures = test_ats_rule_retrieval()
print("ATS rule retrieval failures:", len(ats_failures))
ats_failures


ATS rule retrieval failures: 0


[]

In [31]:
# %%
thresholds = [0.6, 0.7, 0.8, 0.9]
threshold_test_query = "What are ATS rules about using tables in resumes?"

for th in thresholds:
    print(f"\n=== Threshold {th} ===")
    res = search_qdrant_vector(threshold_test_query, top_k=5, score_threshold=th)
    print(f"Returned: {len(res)} results")
    pretty_print_results(res, max_chars=140)



=== Threshold 0.6 ===
Returned: 0 results

=== Threshold 0.7 ===
Returned: 0 results

=== Threshold 0.8 ===
Returned: 0 results

=== Threshold 0.9 ===
Returned: 0 results


In [32]:
# %%
# Simple heuristic reformulator (you can later swap with LLM)
def reformulate_query(query: str) -> str:
    if "ATS" not in query.upper():
        return f"For ATS resume screening: {query}"
    return query

reformulation_queries = [
    "Should I use columns in my resume?",
    "Is it okay to submit a resume as a PNG image?",
    "How many pages should an ATS-friendly resume be?",
]

def compare_reformulation_effect():
    for q in reformulation_queries:
        rq = reformulate_query(q)
        print("\n---")
        print("Original:", q)
        print("Reformulated:", rq)

        orig_res = search_qdrant_vector(q, top_k=3)
        ref_res = search_qdrant_vector(rq, top_k=3)

        print("\nOriginal results:")
        pretty_print_results(orig_res, max_chars=140)

        print("\nReformulated results:")
        pretty_print_results(ref_res, max_chars=140)

compare_reformulation_effect()



---
Original: Should I use columns in my resume?
Reformulated: For ATS resume screening: Should I use columns in my resume?

Original results:
1. score=0.4525 | 'Job titles should be written in a standard format so ATS can correctly parse your experience.'
2. score=0.4482 | 'Save your resume as a PDF unless the job posting specifies otherwise.'
3. score=0.4093 | 'Avoid keyword stuffing in your resume. Overusing the same keyword can be penalized by ATS.'

Reformulated results:
1. score=0.5210 | 'Job titles should be written in a standard format so ATS can correctly parse your experience.'
2. score=0.4902 | 'Keep resume length to one or two pages for optimal ATS scanning.'
3. score=0.4760 | 'Avoid keyword stuffing in your resume. Overusing the same keyword can be penalized by ATS.'

---
Original: Is it okay to submit a resume as a PNG image?
Reformulated: For ATS resume screening: Is it okay to submit a resume as a PNG image?

Original results:
1. score=0.5297 | 'Save your resume as a P

In [33]:
# %%
summary = {
    "embedding_quality_failures": len(embedding_failures),
    "ats_rule_failures": len(ats_failures),
    "chunk_length_stats": chunk_stats,
}

print("=== Retrieval Test Summary ===")
for k, v in summary.items():
    print(f"{k}: {v}")
summary

=== Retrieval Test Summary ===
embedding_quality_failures: 0
ats_rule_failures: 0
chunk_length_stats: {'count': 6, 'min': 64, 'max': 95, 'avg': 79.83333333333333}


{'embedding_quality_failures': 0,
 'ats_rule_failures': 0,
 'chunk_length_stats': {'count': 6,
  'min': 64,
  'max': 95,
  'avg': 79.83333333333333}}